# Scraping de la wiki de Hades 1

Este notebook descarga información de la wiki de Hades (Fandom) con la API de MediaWiki, la limpia, la divide en fragmentos y guarda el resultado en hades1.json.

Mejoras de esta versión: las tablas repiten su cabecera en cada fragmento, se eliminan las frases que mencionan Hades II, se corrigen los apóstrofos, se descartan páginas de parches, se reclasifican entidades y cada fragmento lleva un campo tipo (juego, historia o trivia) para poder filtrar.

El texto de Fandom es CC BY-SA, por lo que cada fragmento guarda su URL y su licencia.

## 1. Instalar las bibliotecas

In [ ]:
!pip install -q requests beautifulsoup4

## 2. Configuración y funciones de descarga y limpieza

Si quieres descartar la Trivia, cambia DESCARTAR_TRIVIA a True.

In [ ]:
import hashlib
import json
import re
import time
import urllib.parse
from collections import Counter
from pathlib import Path

import requests
from bs4 import BeautifulSoup

# ----------------------------------------------------------------------------
# Configuración
# ----------------------------------------------------------------------------
BASE = "https://hades.fandom.com"
API = f"{BASE}/api.php"
HEADERS = {"User-Agent": "Practica-RAG-Hades/1.0 (proyecto educativo de clase)"}

CACHE = Path("cache_hades")
SALIDA = Path("hades1.json")

PAUSA = 0.6            # segundos entre peticiones
MAX_PAGINAS = 600      # tope de seguridad
MAX_PALABRAS = 110     # tamaño máximo de un fragmento
MIN_PALABRAS = 12      # los fragmentos más cortos se descartan
EXCLUIR_HADES2 = True
DESCARTAR_TRIVIA = False   # True: no se guardan los fragmentos de las secciones Trivia
TITULOS_EXCLUIR = re.compile(r"^(Patch\b|Main Page$)")   # páginas que no aportan (notas de parche...)

# Páginas índice: se descargan y se siguen los enlaces internos que contienen.
# Si alguna no existe, el script lo avisa y continúa: ajusta los nombres a tu gusto.
PAGINAS_HUB = [
    "Infernal Arms", "Boons", "Duo Boons", "Legendary Boons", "Keepsakes",
    "Mirror of Night", "Pact of Punishment", "Daedalus Hammer", "Aspects",
    "Enemies", "Bosses", "Resources", "Characters", "Locations",
]

# Páginas que se descargan siempre, aunque no estén enlazadas desde un índice.
PAGINAS_SUELTAS = [
    "Stygian Blade", "Eternal Spear", "Shield of Chaos", "Heart-Seeking Bow",
    "Twin Fists", "Adamant Rail", "Zeus", "Poseidon", "Athena", "Ares",
    "Aphrodite", "Artemis", "Dionysus", "Demeter", "Hermes", "Chaos",
    "Nemesis", "Heat", "Fated List of Minor Prophecies", "Titan Blood", "Darkness",
    "Gemstones", "Chthonic Keys", "Nectar", "Ambrosia", "Diamonds",
    "Tartarus", "Asphodel", "Elysium", "Temple of Styx",
]

# Categorías de la wiki a recorrer (los nombres se pueden ajustar).
CATEGORIAS = ["Boons", "Keepsakes", "Enemies", "Bosses", "Weapon Aspects"]

SECCIONES_IGNORAR = {
    "gallery", "references", "see also", "navigation", "videos", "video",
    "external links", "sources", "notes",
}

# Secciones de historia/personalidad: se marcan con tipo="historia" para poder filtrarlas
SECCIONES_HISTORIA = {
    "physical description", "characteristics and personality", "background", "affinity",
    "description, society and culture", "personality", "appearance", "relationships", "history",
}

# Títulos concretos que se reclasifican (tienen prioridad sobre el mapa de abajo)
TITULO_A_CATEGORIA = {
    "Fated List of Minor Prophecies": "mejora", "Death Defiance": "mejora", "Pom of Power": "mejora",
    "Gameplay mechanics": "mecanica", "Status effects": "mecanica", "Health": "mecanica",
    "Fishing": "mecanica", "Codex": "mecanica", "Chambers and Encounters": "mecanica",
    "Darkness": "recurso", "Nectar": "recurso", "Ambrosia": "recurso", "Titan Blood": "recurso",
    "Gemstones": "recurso", "Chthonic Key": "recurso", "Diamond": "recurso",
    "River Denizens": "enemigo", "Cast": "personaje", "Characters": "personaje",
}

# Clave que se busca en el título y las categorías -> categoría en español
MAPA_CATEGORIA = [
    ("aspect", "aspecto"),
    ("hammer", "martillo"),
    ("duo", "bendicion"),
    ("boon", "bendicion"),
    ("infernal arm", "arma"),
    ("weapon", "arma"),
    ("keepsake", "regalo"),
    ("mirror", "mejora"),
    ("pact", "mejora"),
    ("heat", "mejora"),
    ("boss", "jefe"),
    ("enem", "enemigo"),
    ("resource", "recurso"),
    ("currenc", "recurso"),
    ("location", "zona"),
    ("biome", "zona"),
    ("character", "personaje"),
    ("god", "personaje"),
]

SESION = requests.Session()
SESION.headers.update(HEADERS)


# ----------------------------------------------------------------------------
# Descarga
# ----------------------------------------------------------------------------
def pedir_pagina(titulo):
    """Devuelve {titulo, html, categorias} de una página o None si no existe."""
    CACHE.mkdir(exist_ok=True)
    clave = CACHE / (hashlib.md5(titulo.encode("utf-8")).hexdigest() + ".json")
    if clave.exists():
        return json.loads(clave.read_text(encoding="utf-8"))

    params = {
        "action": "parse", "page": titulo, "prop": "text|categories",
        "redirects": 1, "format": "json", "formatversion": 2,
        "disableeditsection": 1, "disabletoc": 1,
    }
    try:
        r = SESION.get(API, params=params, timeout=30)
        r.raise_for_status()
        datos = r.json()
    except Exception as e:  # red, bloqueo, JSON inválido...
        print(f"  [error] {titulo}: {e}")
        return None
    finally:
        time.sleep(PAUSA)

    if "error" in datos:
        print(f"  [no existe] {titulo}")
        return None

    p = datos["parse"]
    resultado = {
        "titulo": p["title"],
        "html": p["text"],
        "categorias": [c["category"].replace("_", " ") for c in p.get("categories", [])],
    }
    clave.write_text(json.dumps(resultado, ensure_ascii=False), encoding="utf-8")
    return resultado


def miembros_categoria(categoria):
    """Títulos de las páginas (espacio de nombres principal) de una categoría."""
    titulos, continuar = [], None
    while True:
        params = {
            "action": "query", "list": "categorymembers",
            "cmtitle": f"Category:{categoria}", "cmnamespace": 0,
            "cmlimit": 500, "format": "json", "formatversion": 2,
        }
        if continuar:
            params["cmcontinue"] = continuar
        try:
            r = SESION.get(API, params=params, timeout=30)
            r.raise_for_status()
            datos = r.json()
        except Exception as e:
            print(f"  [error] categoría {categoria}: {e}")
            break
        finally:
            time.sleep(PAUSA)
        titulos += [m["title"] for m in datos.get("query", {}).get("categorymembers", [])]
        continuar = datos.get("continue", {}).get("cmcontinue")
        if not continuar:
            break
    if not titulos:
        print(f"  [vacía o no existe] categoría {categoria}")
    return titulos


def enlaces_internos(html):
    sopa = BeautifulSoup(html, "html.parser")
    titulos = []
    for a in sopa.select('a[href^="/wiki/"]'):
        t = urllib.parse.unquote(a["href"][len("/wiki/"):]).split("#")[0].replace("_", " ")
        if t and ":" not in t and t not in titulos:
            titulos.append(t)
    return titulos


def reunir_titulos():
    titulos = list(PAGINAS_SUELTAS)
    for hub in PAGINAS_HUB:
        print("Índice:", hub)
        p = pedir_pagina(hub)
        if p:
            titulos.append(p["titulo"])
            titulos.extend(enlaces_internos(p["html"]))
    for cat in CATEGORIAS:
        print("Categoría:", cat)
        titulos.extend(miembros_categoria(cat))

    vistos, limpios = set(), []
    for t in titulos:
        if t in vistos or "hades ii" in t.lower() or TITULOS_EXCLUIR.search(t):
            continue
        vistos.add(t)
        limpios.append(t)
    return limpios[:MAX_PAGINAS]


# ----------------------------------------------------------------------------
# Limpieza y troceado
# ----------------------------------------------------------------------------
def texto(el):
    t = el.get_text(" ", strip=True)
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"\s+([.,;:!?)\]])", r"\1", t)
    t = re.sub(r"([(\[])\s+", r"\1", t)
    t = re.sub(r"(\w)\s+(['’])(?=\s|s\b|$)", r"\1\2", t)   # "Zagreus ' starting" -> "Zagreus' starting"
    return t.strip()


def limpiar_linea(linea):
    """Quita etiquetas de pestaña y frases que mencionan Hades II."""
    linea = re.sub(r"^(-\s*)?Hades\s*-\s*Hades II\s*", r"\1", linea)
    if not EXCLUIR_HADES2 or "Hades II" not in linea:
        return linea.strip()
    frases = re.split(r"(?<=[.!?])\s+", linea)
    return " ".join(f for f in frases if "Hades II" not in f).strip()


def tipo_fragmento(seccion):
    s = seccion.lower()
    if s == "trivia":
        return "trivia"
    return "historia" if s in SECCIONES_HISTORIA else "juego"


def extraer_secciones(html):
    """Devuelve una lista de (nombre_seccion, [lineas]) a partir del HTML de la página."""
    sopa = BeautifulSoup(html, "html.parser")
    raiz = sopa.select_one("div.mw-parser-output") or sopa

    # Infobox: datos clave en formato "etiqueta: valor"
    datos_generales = []
    for aside in raiz.select("aside.portable-infobox"):
        for d in aside.select(".pi-data"):
            etiqueta = d.select_one(".pi-data-label")
            valor = d.select_one(".pi-data-value")
            if etiqueta and valor:
                linea = limpiar_linea(f"{texto(etiqueta)}: {texto(valor)}")
                if linea:
                    datos_generales.append(linea)
        aside.decompose()

    # Ruido
    for sel in [".navbox", ".toc", "script", "style", "figure", ".gallery",
                "sup.reference", ".reference", ".mw-editsection", ".noprint", "img"]:
        for el in raiz.select(sel):
            el.decompose()

    secciones = [["Resumen", []]]
    if datos_generales:
        secciones.append(["Datos generales", datos_generales])
    actual = secciones[0]

    for el in raiz.select("h2, h3, h4, p, ul, ol, table, dl"):
        # Evita duplicados: ignora elementos dentro de tablas, listas, etc.
        if el.find_parent(["table", "ul", "ol", "li", "dl", "aside"]):
            continue
        if el.name in ("h2", "h3", "h4"):
            nombre = texto(el).replace("[]", "").strip()
            actual = [nombre, []]
            secciones.append(actual)
        elif el.name == "p":
            linea = limpiar_linea(texto(el))
            if linea:
                actual[1].append(linea)
        elif el.name in ("ul", "ol"):
            for li in el.find_all("li", recursive=False):
                linea = limpiar_linea(texto(li))
                if linea:
                    actual[1].append("- " + linea)
        elif el.name == "table":
            cabecera, filas = None, []
            for tr in el.find_all("tr"):
                celdas = [texto(c) for c in tr.find_all(["th", "td"])]
                celdas = [c for c in celdas if c]
                if not celdas:
                    continue
                linea = limpiar_linea(" | ".join(celdas))
                if not linea:
                    continue
                if cabecera is None and not filas and tr.find("th") and not tr.find("td"):
                    cabecera = linea
                else:
                    filas.append(linea)
            if filas:
                actual[1].append(("tabla", cabecera, filas))
        elif el.name == "dl":
            linea = limpiar_linea(texto(el))
            if linea:
                actual[1].append(linea)

    return [(n, l) for n, l in secciones if l]


def trocear(elementos, max_palabras=MAX_PALABRAS):
    """Agrupa líneas en trozos de hasta max_palabras. Las tablas repiten su cabecera en cada trozo."""
    trozos, actual, n = [], [], 0

    def cerrar():
        nonlocal actual, n
        if actual:
            trozos.append("\n".join(actual))
        actual, n = [], 0

    for el in elementos:
        if isinstance(el, tuple):                      # ("tabla", cabecera, filas)
            _, cabecera, filas = el
            cerrar()
            base = 1 if cabecera else 0
            ncab = len(cabecera.split()) if cabecera else 0
            grupo, ng = ([cabecera] if cabecera else []), ncab
            for fila in filas:
                nf = len(fila.split())
                if len(grupo) > base and ng + nf > max_palabras:
                    trozos.append("\n".join(grupo))
                    grupo, ng = ([cabecera] if cabecera else []), ncab
                grupo.append(fila)
                ng += nf
            if len(grupo) > base:
                trozos.append("\n".join(grupo))
            continue

        frases = re.split(r"(?<=[.!?])\s+", el) if len(el.split()) > max_palabras else [el]
        for f in frases:
            nf = len(f.split())
            if actual and n + nf > max_palabras:
                cerrar()
            actual.append(f)
            n += nf
    cerrar()
    return trozos


def clasificar(titulo, categorias):
    if titulo in TITULO_A_CATEGORIA:
        return TITULO_A_CATEGORIA[titulo]
    base = " ".join([titulo] + categorias).lower()
    for clave, nombre in MAPA_CATEGORIA:
        if clave in base:
            return nombre
    return "general"


def es_solo_hades2(categorias):
    tiene_h2 = any("hades ii" in c.lower() for c in categorias)
    tiene_h1 = any(re.search(r"\bhades\b(?!\s+ii)", c.lower()) for c in categorias)
    return tiene_h2 and not tiene_h1


## 3. Construcción de los fragmentos

In [ ]:
# ----------------------------------------------------------------------------
# Programa principal
# ----------------------------------------------------------------------------
def construir_fragmentos():
    titulos = reunir_titulos()
    print(f"\nPáginas candidatas: {len(titulos)}\n")

    fragmentos, textos_vistos, excluidas = [], set(), []
    for i, titulo in enumerate(titulos, start=1):
        pagina = pedir_pagina(titulo)
        if not pagina:
            continue
        if EXCLUIR_HADES2 and es_solo_hades2(pagina["categorias"]):
            excluidas.append(pagina["titulo"])
            continue

        entidad = pagina["titulo"]
        categoria = clasificar(entidad, pagina["categorias"])
        url = f"{BASE}/wiki/{urllib.parse.quote(entidad.replace(' ', '_'))}"

        for nombre, lineas in extraer_secciones(pagina["html"]):
            if nombre.lower() in SECCIONES_IGNORAR:
                continue
            if EXCLUIR_HADES2 and "hades ii" in nombre.lower():
                continue
            tipo = tipo_fragmento(nombre)
            if DESCARTAR_TRIVIA and tipo == "trivia":
                continue
            for trozo in trocear(lineas):
                if len(trozo.split()) < MIN_PALABRAS:
                    continue
                cabecera = entidad if nombre in ("Resumen", entidad) else f"{entidad} - {nombre}"
                cuerpo = f"{cabecera}: {trozo}"
                if cuerpo in textos_vistos:
                    continue
                textos_vistos.add(cuerpo)
                fragmentos.append({
                    "id": len(fragmentos),
                    "juego": "hades1",
                    "categoria": categoria,
                    "tipo": tipo,
                    "entidad": entidad,
                    "seccion": nombre,
                    "fuente": "hades.fandom.com",
                    "url": url,
                    "wiki_categorias": pagina["categorias"],
                    "licencia": "CC BY-SA (contenido de Fandom)",
                    "texto": cuerpo,
                })
        if i % 25 == 0:
            print(f"  procesadas {i}/{len(titulos)} páginas, {len(fragmentos)} fragmentos")

    return fragmentos, excluidas


## 4. Ejecutar el scraping y guardar hades1.json

La primera ejecución puede tardar varios minutos por la pausa entre peticiones. Si repites la ejecución en la misma sesión, la caché evita volver a descargar.

In [ ]:
fragmentos, excluidas = construir_fragmentos()
SALIDA.write_text(json.dumps(fragmentos, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nGuardado {SALIDA} con {len(fragmentos)} fragmentos")
print("Por categoría:", dict(Counter(f["categoria"] for f in fragmentos)))
print("Por tipo:", dict(Counter(f["tipo"] for f in fragmentos)))
print("Fragmentos que aún mencionan Hades II:", sum("Hades II" in f["texto"] for f in fragmentos))
print("Páginas excluidas por ser solo de Hades II:", len(excluidas))
print("Entidades con más fragmentos:",
      Counter(f["entidad"] for f in fragmentos).most_common(10))


Índice: Infernal Arms
Índice: Boons
Índice: Duo Boons
Índice: Legendary Boons
Índice: Keepsakes
Índice: Mirror of Night
Índice: Pact of Punishment
Índice: Daedalus Hammer
Índice: Aspects
  [no existe] Aspects
Índice: Enemies
  [no existe] Enemies
Índice: Bosses
  [no existe] Bosses
Índice: Resources
  [no existe] Resources
Índice: Characters
Índice: Locations
  [no existe] Locations
Categoría: Boons
Categoría: Keepsakes
Categoría: Enemies
Categoría: Bosses
Categoría: Weapon Aspects
  [vacía o no existe] categoría Weapon Aspects

Páginas candidatas: 408

  [no existe] Chthonic Keys
  procesadas 25/408 páginas, 422 fragmentos
  [no existe] Diamonds
  procesadas 50/408 páginas, 821 fragmentos
  [no existe] Chtonic Key
  procesadas 75/408 páginas, 1037 fragmentos
  procesadas 100/408 páginas, 1330 fragmentos
  procesadas 125/408 páginas, 1676 fragmentos
  [no existe] Greeneries
  [no existe] Indulgences
  procesadas 225/408 páginas, 1764 fragmentos
  procesadas 250/408 páginas, 1836 fragme

## 5. Revisar el resultado

In [ ]:
import pandas as pd

df = pd.read_json('hades1.json')
print(df.shape)
df[['id', 'categoria', 'tipo', 'entidad', 'seccion']].head(15)

(1864, 11)


,id,categoria,tipo,entidad,seccion
0,0,arma,juego,Stygian Blade,Resumen
1,1,arma,juego,Stygian Blade,Fighting style
2,2,arma,juego,Stygian Blade,Statistics
3,3,arma,juego,Stygian Blade,Aspects
4,4,arma,juego,Stygian Blade,Aspects
5,5,arma,juego,Stygian Blade,Aspects
6,6,arma,juego,Stygian Blade,Aspects
7,7,arma,juego,Stygian Blade,Aspects
8,8,arma,juego,Stygian Blade,Revealing the Aspect of Arthur
9,9,arma,juego,Stygian Blade,Aspect of Arthur Moveset


In [ ]:
# Descargar el archivo a tu ordenador
from google.colab import files
files.download('hades1.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>